# Unit 2 Assignment: Mixture of Experts (MoE) Router

**Smart Customer Support Router** using a Mixture of Experts architecture with the Groq API.

The system routes user queries to specialized experts:
- **Technical Expert** — bug reports, code errors
- **Billing Expert** — refunds, charges, subscriptions
- **Sales Expert** — new inquiries, pricing questions
- **Tool Use Expert** *(Bonus)* — real-time data lookups (e.g., crypto prices)
- **General Expert** — fallback for casual chat

## 1. Setup — Install Dependencies

In [1]:
%pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 7.7 MB/s eta 0:00:00


In [3]:
import os, getpass
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# fallback: prompt for key if .env is missing / placeholder
if not os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY") == "your_groq_api_key_here":
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client ready!")

Groq client ready!


## 2. Define the Experts

Each expert uses the **same base model** (`llama-3.1-8b-instant`) but with a different **system prompt** to specialize its behaviour.

In [11]:
BASE_MODEL = "llama-3.1-8b-instant"

MODEL_CONFIG = {
    "technical": {
        "model": BASE_MODEL,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Senior Technical Support Engineer. "
            "You are rigorous, code-focused, and precise. "
            "When the user reports a bug or error, ask clarifying questions if needed, "
            "then provide a clear, step-by-step fix with code snippets. "
            "Always explain *why* the error happened."
        ),
    },
    "billing": {
        "model": BASE_MODEL,
        "temperature": 0.7,
        "system_prompt": (
            "You are a Billing & Accounts Support Specialist. "
            "You are empathetic, financial-focused, and policy-driven. "
            "Help the user understand charges, refunds, and subscription policies. "
            "If a refund may be warranted, outline the steps clearly. "
            "Always be polite and reassuring."
        ),
    },
    "sales": {
        "model": BASE_MODEL,
        "temperature": 0.7,
        "system_prompt": (
            "You are a friendly Sales Representative. "
            "Help the user discover the right product or plan for their needs. "
            "Highlight benefits, compare options, and guide them toward a purchase. "
            "Be enthusiastic but not pushy."
        ),
    },
    "tool_use": {
        "model": BASE_MODEL,
        "temperature": 0.0,
        "system_prompt": (
            "You are a Data Assistant with access to real-time tools. "
            "When a tool result is provided, use it to give a concise, accurate answer."
        ),
    },
    "general": {
        "model": BASE_MODEL,
        "temperature": 0.7,
        "system_prompt": (
            "You are a friendly, general-purpose customer support assistant. "
            "Help with any casual question, small talk, or general inquiry. "
            "Be warm and helpful."
        ),
    },
}

print("Expert configs defined:")
for name in MODEL_CONFIG:
    print(f"   • {name}")

Expert configs defined:
   • technical
   • billing
   • sales
   • tool_use
   • general


## 3. [Bonus] Mock Tool for Real-Time Data

A simple mock function that pretends to fetch live crypto prices. The **tool_use** expert will use this.

In [12]:
import random

def mock_get_crypto_price(coin: str) -> str:
    """Simulates fetching a live crypto price."""
    prices = {
        "bitcoin":  round(random.uniform(60000, 70000), 2),
        "ethereum": round(random.uniform(3000, 4000), 2),
        "solana":   round(random.uniform(120, 180), 2),
        "dogecoin": round(random.uniform(0.05, 0.15), 4),
    }
    coin = coin.lower().strip()
    if coin in prices:
        return f"The current price of {coin.title()} is ${prices[coin]:,.2f} USD."
    return f"Sorry, I don't have price data for '{coin}'."

# quick test
print(mock_get_crypto_price("bitcoin"))

The current price of Bitcoin is $66,708.42 USD.


## 4. The Router - Intent Classification

The router uses an LLM call with `temperature=0` (for deterministic output) to classify user intent into one of the expert categories.

In [13]:
VALID_CATEGORIES = list(MODEL_CONFIG.keys())

def route_prompt(user_input: str) -> str:
    """
    Uses an LLM call to classify the user's intent.
    Returns ONE category name: technical | billing | sales | tool_use | general
    """
    routing_prompt = (
        "You are a classification bot. "
        "Classify the following user message into exactly ONE of these categories:\n"
        f"  {VALID_CATEGORIES}\n\n"
        "Rules:\n"
        "- 'technical'  → bug reports, code errors, software issues\n"
        "- 'billing'    → charges, refunds, invoices, subscriptions\n"
        "- 'sales'      → product inquiries, pricing, plan comparisons\n"
        "- 'tool_use'   → requests for real-time data like crypto/stock prices\n"
        "- 'general'    → anything else (casual chat, greetings, etc.)\n\n"
        "Return ONLY the category word. Nothing else."
    )

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,          # deterministic
        max_tokens=10,          # we only need one word
        messages=[
            {"role": "system", "content": routing_prompt},
            {"role": "user",   "content": user_input},
        ],
    )

    category = response.choices[0].message.content.strip().lower()

    # safety check — fall back to 'general' if LLM returns something unexpected
    if category not in VALID_CATEGORIES:
        print(f"Router returned unknown category '{category}', falling back to 'general'")
        category = "general"

    return category

# quick test
test = "My python script is throwing an IndexError on line 5."
print(f"Query  : {test}")
print(f"Routed : {route_prompt(test)}")

Query  : My python script is throwing an IndexError on line 5.
Routed : technical


## 5. The Orchestrator

`process_request` ties everything together:
1. **Route** the query to decide the category.
2. **Select** the right expert config (system prompt + temperature).
3. If `tool_use`, run the mock tool first and inject its result.
4. **Call** the LLM with the expert persona and return the answer.

In [14]:
import re

def _extract_coin(text: str) -> str:
    """Try to pull a coin name out of the user's message."""
    known = ["bitcoin", "ethereum", "solana", "dogecoin"]
    for coin in known:
        if coin in text.lower():
            return coin
    # fallback: grab the last noun-ish word near 'price'
    match = re.search(r"price\s+of\s+(\w+)", text, re.IGNORECASE)
    return match.group(1) if match else "bitcoin"


def process_request(user_input: str) -> str:
    """
    Main orchestrator:
      1. Route the query
      2. Pick the expert config
      3. (bonus) Run tool if needed
      4. Call the LLM with the expert system prompt
    """
    # --- Step 1: Route ---
    category = route_prompt(user_input)
    config   = MODEL_CONFIG[category]

    print(f"Router decision : {category}")
    print(f"Model           : {config['model']}")
    print(f"Temperature     : {config['temperature']}")

    # --- Step 2 (bonus): Handle tool_use ---
    if category == "tool_use":
        coin       = _extract_coin(user_input)
        tool_result = mock_get_crypto_price(coin)
        print(f"Tool result     : {tool_result}")
        # inject tool output into the user message so the LLM can reference it
        user_input = f"{user_input}\n\n[Tool Result]: {tool_result}"

    # --- Step 3: Call the expert ---
    response = client.chat.completions.create(
        model=config["model"],
        temperature=config["temperature"],
        max_tokens=512,
        messages=[
            {"role": "system", "content": config["system_prompt"]},
            {"role": "user",   "content": user_input},
        ],
    )

    return response.choices[0].message.content

print("Orchestrator ready!")

Orchestrator ready!


## 6. Testing

In [15]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "What plans do you offer for small businesses?",
    "What is the current price of Bitcoin?",
    "Hey, how are you doing today?",
]

for query in test_queries:
    print("=" * 70)
    print(f"User: {query}\n")
    answer = process_request(query)
    print(f"\nExpert:\n{answer}\n")

User: My python script is throwing an IndexError on line 5.

Router decision : technical
Model           : llama-3.1-8b-instant
Temperature     : 0.7

Expert:
An `IndexError` typically occurs when you're trying to access an element in a list or other sequence that doesn't exist. To help you troubleshoot, I'll need more information.

Can you please provide the following:

1. The exact error message (including the line number and any relevant details).
2. The code snippet that's causing the issue (lines 1-10 should be enough).
3. The input data that's being used when the error occurs (if applicable).

Additionally, could you tell me:

* What you're trying to do with your script?
* What line 5 of your script looks like?

With this information, I'll do my best to guide you through the fix.

**Example of what I'm looking for:**

Error message:
```
IndexError: list index out of range (line 5)
```
Code snippet:
```python
my_list = [1, 2, 3]
# Your code here...
```
Input data:
```python
input_

## 7. Interactive Mode (optional)

Run the cell below to chat interactively. Type `quit` to stop.

In [16]:
while True:
    user_input = input("\nYou: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        print("Bye!")
        break
    answer = process_request(user_input)
    print(f"\nExpert:\n{answer}")

Router decision : technical
Model           : llama-3.1-8b-instant
Temperature     : 0.7

Expert:
List comprehension in Python is a concise way to create lists. The basic syntax is as follows:

```python
new_list = [expression for element in iterable if condition]
```

Here's a breakdown of the components:

- `new_list`: The new list that will be created.
- `expression`: The operation you want to perform on each element in the iterable.
- `element`: The individual element being processed in the iterable.
- `iterable`: The list, tuple, or other iterable that contains the elements to be processed.
- `condition`: An optional filter that can be applied to each element.

Let's consider an example:

```python
numbers = [1, 2, 3, 4, 5]
squared_numbers = [x**2 for x in numbers]
print(squared_numbers)  # Output: [1, 4, 9, 16, 25]
```

In this example, we create a new list `squared_numbers` by squaring each number in the `numbers` list.

Here's another example with a condition:

```python
number